# 43 · Revision of the zonation classes: reliability ceilings, symmetric calls, pathway reframing

This notebook answers the internal referee review (`docs/paper/referee_report_v01.md`, points
M2, M3, M7, D2 and D3). Every contrast uses segment labels. The labels are a parameter: by
default they are the reviewed labels, and workstream 1's independently validated labels can be
swapped in (see section 1).

## Pre-registered protocol (fixed before any result in this notebook was computed)

### Part 1 · D2: same-species reliability ceiling and a conservation index

**Gene gradients** come from S2 − S1 and S3 − early, where early is S1 + S2. In our data, the
mouse also gets the region-matched S3c − early, which uses only cortical-like mouse S3.

| Species | Dataset | Replicates |
|---|---|---|
| Human | ours | 2 sections |
| Human | Lake/KPMP healthy cortex | 7 donors |
| Human | Census human cortex | 6 donors; S3 − early only |
| Mouse | ours | 2 mice |
| Mouse | Census snRNA male / female | 12 donors each |
| Mouse | microdissection male (GSE150338) | mice, unpaired across segments |
| Mouse | microdissection by sex (GSE212213) | means only, so no noise estimate |
| Rat | microdissection (GSE56743) | 7 per segment, unpaired; RPKM |

**Correlation.**
- Pearson correlation of gene gradients, each winsorized at the 1st and 99th percentiles. Genes
  must be expressed in both datasets: ≥ 20 counts in every relevant pseudobulk for our data, and
  mean log2 level ≥ 3 in one segment group externally.
- The correlation is **disattenuated** by each dataset's replicate reliability (signal ÷ total
  variance across genes). Datasets without replicates are reported raw only.

**Conservation index (primary).**
- Index = r*(our human, our mouse) ÷ √(r*(our human, Lake) × r*(our mouse, mouse snRNA male)).
  It is computed on genes expressed in all four datasets, with 95% intervals from 1,000 gene
  bootstraps.
- An external-only version replaces the numerator with r*(Lake, mouse snRNA male).

**Decision.**

| Upper 95% bound of the index, S2 − S1 and S3 − early | Wording |
|---|---|
| < 0.5 in both contrasts | "largely different genes" stands |
| the index lies between 0.5 and 0.8 | "partly conserved" |
| ≥ 0.8 | "largely conserved; apparent divergence is a cross-dataset ceiling" |

**Rat.** Rat is used only if its replicate reliability is ≥ 0.5 for the contrast and ≥ 70% of our
gene space matches by symbol. Otherwise it is excluded as unreliable, and rat-based arguments are
withdrawn.

### Part 2 · D3: symmetric classification

The rules are notebook 40's (moderated per-species calls, reference confirmation of zonated
calls), plus the following settings:

| Setting | Change |
|---|---|
| v1 | notebook 40's primary |
| A3b amplitude-matched | human effect floor **and** flat margin multiplied by the contrast's noise-corrected human ÷ mouse amplitude |
| A8 external flat confirmation | a species-only class also needs the other species' atlas to be flat-compatible: not significant in the same direction (one-sided p > 0.05), and |mean| < 0.5. Otherwise the gene is indeterminate. |
| **Symmetric (revision primary)** | A3b + A8 |
| v1 + A6 / A7, Symmetric + A6 / A7 | mouse confirmed by female snRNA or female microdissection; for A6 the flat check also uses female snRNA |

Class counts are reported side by side for S2 − S1, S3c − early, S3 − early and the union,
probe-balanced genes only. We also count mouse-only genes with |mouse| ≥ 1.5 and human within
±0.3.

**Decision on asymmetry**, from the mouse-only ÷ human-only ratio under the symmetric setting:

| Ratio ≥ 2 in … | Conclusion |
|---|---|
| S2 − S1 and S3c − early | asymmetry supported |
| S2 − S1 only | asymmetry confined to S2 − S1 |
| neither | no consistent asymmetry |

### Part 3 · M7: what the pathway layer summarises

**Member categories.** Each robust pathway's contributing members (the top 10% by notebook 12's
T_spatial) are grouped under the symmetric setting, over S2 − S1 and S3c − early:

| Category | Mouse | Human |
|---|---|---|
| mouse-led | zonated in some contrast | not zonated there, and the gene is zonated in both species in no contrast |
| human-led | not zonated there | zonated in some contrast |
| shared | conserved or reversal in some contrast | conserved or reversal in some contrast |

**Pathway labels.** Among a pathway's members zonated in either species (at least 3 needed):
- ≥ 50% mouse-led: a **summary of mouse zonation**;
- ≥ 50% shared: **shared**;
- ≥ 50% human-led: **human-led**;
- otherwise **mixed**.

A pathway with fewer than 3 such members is **unresolved**.

**Probe rule.** A robust pathway failing notebook 37's equal-probe check is flagged as
probe-sensitive. A "beyond flattening" pathway from notebook 39 that is probe-sensitive is
dropped from that claim.

## 1 · Folders, label source and settings

**Labels.** By default, the reviewed S1/S2/S3 labels are used. To rerun with other labels, pass
`--labels file.csv` or set `PSEUDOSPACE_SEGMENT_LABELS`. The file needs columns `structure`
(notebook 13 structure ids) and `segment` (S1, S2, S3, or PT-S1 …). Without either, the notebook
uses `results/pt_revision_labels/segment_labels.csv` when that file exists. Structures without a
label are left out of every segment contrast. Results go to
`results/pt_revision_classes/<label tag>/`.

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--labels', type=Path, help='CSV with columns structure, segment')
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
label_source = args.labels or os.environ.get('PSEUDOSPACE_SEGMENT_LABELS')
default_labels = results_root / 'pt_revision_labels' / 'segment_labels.csv'
label_file = Path(label_source).expanduser() if label_source else (default_labels if default_labels.is_file() else None)
LABEL_TAG = label_file.stem if label_file else 'reviewed'
upstream13 = results_root / 'minimal_pt_scfates'
upstream37 = results_root / 'pt_pathway_final' / 'scfates'
external = data_root / 'external'
output = results_root / 'pt_revision_classes' / LABEL_TAG
figures, tables = output / 'figures', output / 'tables'
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
probe_file = results_root / 'pt_literature_deep' / 'probe_counts.csv'
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad',
            upstream37 / 'tables' / 'table_S2_pathways.csv',
            results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv',
            results_root / 'pt_pathway_remodeling_scfates' / 'pathway_coverage.csv',
            results_root / 'pt_zonation_amplitude' / 'tables' / 'robust_pathway_flattening_classes.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz',
            external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz', lake_file, probe_file]
required += [label_file] if label_file else []
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '43.revision_classes.1'
SEED, N_BOOT = 43, 1000
MIN_COUNTS, MIN_NUCLEI, REFERENCE_LEVEL = 20, 50, 3.
ALPHA, EFFECT, MARGIN, CONFIRM_ALPHA = .05, .5, .3, .05
INDEX_STRONG, INDEX_PARTIAL = .5, .8
RAT_RELIABILITY, RAT_MAPPED = .5, .7
print('Labels:', label_file or 'reviewed labels (notebook 13)', '· results:', output)

## 2 · Our structures, counts and segment pseudobulks (as notebook 39; labels from section 1)

- **Cross-species data.** All of notebook 13's PT structures with their reviewed S1/S2/S3 labels.
  No coordinate filter is applied.
- **Gene space.** Accepted orthologs measured in all four specimens (mouse symbols).
- **Library size.** The total over that gene space.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.stage_cache import cached_payload, cached_frame, digest
from pseudospace import zonation_amplitude as amplitude_module
from pseudospace.zonation_amplitude import (group_sums, log_ratio, ols_slope, deming_slope, amplitude_ratio,
                                            bootstrap_ratio, flattening_class)

cache = output / 'stage_cache'
from pseudospace import zonation_classes as _classes_module, zonation_reliability as _reliability_module
code = digest([NOTEBOOK_LOGIC_VERSION, Path(amplitude_module.__file__), Path(_classes_module.__file__), Path(_reliability_module.__file__)])
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in names}
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
measured = adata.var.measured_in_both_inputs.to_numpy()
gene_space = adata.var_names[measured]
counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
lognorm = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)[:, measured]
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
if label_file is not None:
    supplied = pd.read_csv(label_file)
    if not {'structure', 'segment'}.issubset(supplied) or supplied.structure.duplicated().any():
        raise ValueError('The label file needs unique `structure` ids and a `segment` column.')
    mapped_labels = (supplied.set_index(supplied.structure.astype(str)).segment.astype(str)
                     .str.replace('PT-', '', regex=False).reindex(adata.obs_names.astype(str)))
    print(f'{int(mapped_labels.notna().sum()):,} of {len(mapped_labels):,} structures carry a supplied label; '
          f'agreement with the reviewed labels: {(mapped_labels.to_numpy() == segment).mean():.1%}')
    segment = mapped_labels.where(mapped_labels.isin(['S1', 'S2', 'S3']), 'unlabelled').to_numpy()
library = np.asarray(counts.sum(axis=1)).ravel()
display(pd.crosstab(specimen, segment))
structure_depth = pd.DataFrame({'specimen': specimen, 'library': library}).groupby('specimen').library.describe()[['count', '50%', 'mean']]
display(structure_depth.rename(columns={'50%': 'median counts per structure'}))

### Cortical-like mouse S3

Depth is the mean distance to the three nearest reviewed glomeruli of the same specimen, with
centroids verified against the fine GeoJSON (notebook 33's rule).

In [ ]:
from scipy.spatial import cKDTree
from pseudospace.repeated_inputs import _read_and_validate_segmentation


def mouse_depth():
    pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
    anchors = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
    pass1.file.close()
    values = np.full(len(specimen), np.nan)
    for name in expected['mouse']:
        native = ad.read_h5ad(source_paths[name], backed='r')
        source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
        native.file.close()
        mask = specimen == name
        structures = source_obs.loc[adata.obs_names[mask].str[len(name) + 1:]]
        _read_and_validate_segmentation(data_root, name, structures)
        glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
        _read_and_validate_segmentation(data_root, name, glomeruli)
        distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
            structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
        values[mask] = distance.mean(axis=1)
    return {'depth': values}


depth = cached_payload('mouse_depth', mouse_depth, root=cache, inputs={'obs': list(adata.obs_names)}, code=code)['depth']
cortical_s3 = np.zeros(len(specimen), bool)
for name in expected['mouse']:
    limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
    cortical_s3 |= (specimen == name) & (segment == 'S3') & (depth <= limit)
segment_region = np.where((segment == 'S3') & np.isin(specimen, expected['mouse']) & ~cortical_s3, 'S3deep', segment)
print('Cortical-like mouse S3 structures:', {n: int((cortical_s3 & (specimen == n)).sum()) for n in expected['mouse']})

In [ ]:
def specimen_pseudobulk(count_matrix, specimen_labels, segment_labels, genes):
    """Summed counts per specimen and segment (S1, S2, S3, cortical-S3 and early = S1 + S2)."""
    names_, sums = group_sums(count_matrix, np.char.add(np.char.add(specimen_labels.astype(str), '|'), segment_labels.astype(str)))
    frame = pd.DataFrame(sums, index=names_, columns=genes)
    out = {}
    for name in np.unique(specimen_labels):
        rows = lambda labels: [f'{name}|{s}' for s in labels if f'{name}|{s}' in frame.index]
        out[(name, 'S1')] = frame.loc[rows(['S1'])].sum()
        out[(name, 'S2')] = frame.loc[rows(['S2'])].sum()
        out[(name, 'S3')] = frame.loc[rows(['S3', 'S3deep'])].sum()
        out[(name, 'S3c')] = frame.loc[rows(['S3'])].sum()
        out[(name, 'early')] = out[(name, 'S1')] + out[(name, 'S2')]
    return pd.DataFrame(out)


ours_counts = specimen_pseudobulk(counts, specimen, segment_region, gene_space)
ours_library = pd.Series({key: float(library[(specimen == key[0]) & np.isin(
    segment_region, {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}[key[1]])].sum())
    for key in ours_counts.columns})

## 3 · Our gradients

- **Pseudobulk log2 ratio (primary):** log2 CPM(late) − log2 CPM(early) from summed counts per
  specimen.
- **log1p mean (sensitivity):** mean log1p(CP10k) per segment, the contrast used in notebooks 34
  and 37.
- **Contrasts:**
  - `S2-S1`;
  - `S3c-early`, with mouse S3 restricted to cortical-like structures and human S3 complete (both
    human sections are cortex);
  - `S3-early`, all S3.

In [ ]:
CONTRASTS = {'S2-S1': ('S2', 'S1'), 'S3c-early': ('S3c', 'early'), 'S3-early': ('S3', 'early')}


def gradients(pb, pb_library, specimens):
    out = {}
    for contrast, (late, early) in CONTRASTS.items():
        for name in specimens:
            out[(contrast, name)] = log_ratio(pb[(name, late)], pb_library[(name, late)], pb[(name, early)], pb_library[(name, early)])
    return pd.DataFrame(out, index=pb.index)


ours = gradients(ours_counts, ours_library, names)
log1p_means = {}
for name in names:
    for label, parts in {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}.items():
        rows = (specimen == name) & np.isin(segment_region, parts)
        log1p_means[(name, label)] = np.asarray(lognorm[rows].mean(axis=0)).ravel()
log1p_means = pd.DataFrame(log1p_means, index=gene_space)
ours_log1p = pd.DataFrame({(contrast, name): log1p_means[(name, late)] - log1p_means[(name, early)]
                           for contrast, (late, early) in CONTRASTS.items() for name in names})


def count_filter(pb, specimens, parts):
    return pb[[(n, p) for n in specimens for p in parts]].min(axis=1).ge(MIN_COUNTS)


our_genes = {contrast: gene_space[count_filter(ours_counts, names, list(parts)).to_numpy()]
             for contrast, parts in CONTRASTS.items()}
def species_mean(frame, contrast, specimens):
    return frame[[(contrast, n) for n in specimens]].mean(axis=1)


print({c: len(g) for c, g in our_genes.items()}, 'genes pass the count filter (cross-species)')

## 4 · Independent references

Each reference gives a gene-level gradient for each contrast (log2 units), plus a donor-resampling
function for the bootstrap.
- **Microdissection male** (GSE150338): mean log2(TPM + 1) per mouse, then over mice. Segments
  come from different mice, so the bootstrap resamples mice within each segment.
- **Microdissection by sex** (GSE212213): segment means only, so the bootstrap resamples genes only.
- **Mouse snRNA** (Census): log2(CPM + 1) per donor; early = pooled S1 + S2 counts; donors with
  ≥ 50 nuclei in every segment.
- **Lake / KPMP human cortex** (CELLxGENE v1.5): PT-S1/S2/S3 nuclei of cortex samples, summed per
  donor and segment, with genes mapped by Ensembl id. Healthy donors are the human reference. All
  donors, healthy plus AKI and CKD, enter the tissue-state analysis.

For microdissection, "S3c" is the dissected S3, which lies in the outer stripe, so it equals
"S3".

In [ ]:
import re


def microdissection_male():
    table = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
    table = table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
    per_mouse = {}
    for column in table.columns:
        match = re.fullmatch(r'(m\d+)_PTS([123])_[a-z]', column)
        if match:
            per_mouse.setdefault((f'S{match.group(2)}', match.group(1)), []).append(column)
    values = {key: np.log2(table[cols].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1).mean(axis=1)
              for key, cols in per_mouse.items()}
    return pd.DataFrame(values).reindex(gene_space)


micro_male = microdissection_male()
micro_sex_table = pd.read_csv(external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz', sep='\t')
micro_sex_table = micro_sex_table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
micro_sex = {sex: pd.DataFrame({f'S{k}': np.log2(pd.to_numeric(micro_sex_table[f'{prefix}PTS{k}'], errors='coerce').clip(lower=0) + 1)
                                for k in (1, 2, 3)}).reindex(gene_space)
             for sex, prefix in (('male', ''), ('female', 'f'))}


def census_segments(label, segment_map):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_NUCLEI)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    counts_out, totals, info = {}, {}, {}
    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        for seg_name in needed:
            index = block.index[block.segment.eq(seg_name)]
            counts_out[(donor, seg_name)] = table.loc[index].sum()
            totals[(donor, seg_name)] = meta.loc[index, 'total_counts_all_genes'].sum()
        info[donor] = sex
    return pd.DataFrame(counts_out).reindex(gene_space).fillna(0), pd.Series(totals), pd.Series(info)


sn_counts, sn_totals, sn_sex = census_segments('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                          'epithelial cell of proximal tubule segment 2': 'S2',
                                                          'epithelial cell of proximal tubule segment 3': 'S3'})
census_h_counts, census_h_totals, census_h_sex = census_segments('human', {
    'kidney proximal convoluted tubule epithelial cell': 'early', 'epithelial cell of proximal tubule segment 3': 'S3'})


def lake_pseudobulk():
    """Lake/KPMP cortex PT nuclei summed per donor x disease x segment, plus PT-state counts per donor."""
    import h5py
    from pseudospace.replication_rules import csr_group_sums

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        cortex = obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
        keep = cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3'])
        label = (obs.donor_id + '|' + obs.disease + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
        states = obs[cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT'])]
        state_counts = states.groupby(['donor_id', 'subclass.l2']).size().unstack(fill_value=0)
    human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T.reindex(columns=gene_space, fill_value=0.)
    state_columns = ['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT']
    state_counts = state_counts.reindex(columns=state_columns, fill_value=0)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': frame.to_numpy(),
            'state_donors': state_counts.index.to_numpy().astype(str), 'state_counts': state_counts.to_numpy()}


lake = cached_payload('lake_pseudobulk', lake_pseudobulk, root=cache,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'all'},
                      inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'genes': list(gene_space),
                              'orthologs': digest(orthologs)}, code=code)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'disease', 'sex', 'segment'])
lake_meta['nuclei'], lake_meta['total'] = lake['nuclei'], lake['totals']
lake_sums = pd.DataFrame(lake['sums'], columns=gene_space)
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_NUCLEI).all(axis=1)]
lake_disease = lake_meta.drop_duplicates('donor').set_index('donor').disease
lake_states = pd.DataFrame(lake['state_counts'], index=lake['state_donors'], columns=['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT'])
lake_counts = {}
lake_totals = {}
for donor in lake_donors:
    for seg_name in ('S1', 'S2', 'S3'):
        rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.eq(seg_name)]
        lake_counts[(donor, seg_name)] = lake_sums.loc[rows].sum()
        lake_totals[(donor, seg_name)] = float(lake_meta.loc[rows, 'total'].sum())
lake_counts, lake_totals = pd.DataFrame(lake_counts), pd.Series(lake_totals)
print(f'Lake cortex donors with ≥ {MIN_NUCLEI} nuclei in each segment: {len(lake_donors)} '
      f'({lake_disease.loc[lake_donors].value_counts().to_dict()}); mouse snRNA donors: {sn_sex.value_counts().to_dict()}')

In [ ]:
def log2_cpm(counts_series, total):
    return np.log2(counts_series / total * 1e6 + 1)


def donor_contrasts(count_frame, totals, donors, contrast):
    """Per-donor log2 CPM contrast and per-segment levels; early pools S1 + S2 counts."""
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    rows_contrast, rows_level = {}, {}
    for donor in donors:
        def level(parts):
            pooled = sum(count_frame[(donor, p)] for p in parts)
            return log2_cpm(pooled, sum(totals[(donor, p)] for p in parts))
        late_level = level([late])
        early_level = level(['S1', 'S2'] if early == 'early' else [early])
        rows_contrast[donor] = late_level - early_level
        rows_level[donor] = np.maximum(late_level, early_level)
    return pd.DataFrame(rows_contrast).T, pd.DataFrame(rows_level).T


def reference_from_donors(count_frame, totals, donors, contrast):
    """Reference gradient (mean over donors), expressed-gene mask and a donor-resampling draw function."""
    return donor_reference(*donor_contrasts(count_frame, totals, donors, contrast))


def reference_from_census_human():
    """Census human cortex: S3 minus convoluted PT per donor (only for the S3 contrasts)."""
    donors = census_h_sex.index
    late = {d: log2_cpm(census_h_counts[(d, 'S3')], census_h_totals[(d, 'S3')]) for d in donors}
    early = {d: log2_cpm(census_h_counts[(d, 'early')], census_h_totals[(d, 'early')]) for d in donors}
    per_donor = pd.DataFrame({d: late[d] - early[d] for d in donors}).T
    level = pd.DataFrame({d: np.maximum(late[d], early[d]) for d in donors}).T
    return donor_reference(per_donor, level)


def donor_reference(per_donor, level):
    expressed = level.mean().ge(REFERENCE_LEVEL)

    def draw(rng):
        rows = np.arange(len(per_donor)) if rng is None else rng.integers(0, len(per_donor), len(per_donor))
        return per_donor.iloc[rows].mean()[expressed]

    error_variance = float((per_donor.loc[:, expressed].var(ddof=1) / len(per_donor)).mean())
    return {'gradient': per_donor.mean()[expressed], 'draw': draw, 'error_variance': error_variance,
            'n_donors': len(per_donor), 'per_donor': per_donor}


def reference_from_micro_male(contrast):
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    parts_early = ['S1', 'S2'] if early == 'early' else [early]
    by_segment = {s: micro_male[[c for c in micro_male.columns if c[0] == s]] for s in ('S1', 'S2', 'S3')}

    def gradient(rng):
        means = {}
        for s, frame in by_segment.items():
            pick = np.arange(frame.shape[1]) if rng is None else rng.integers(0, frame.shape[1], frame.shape[1])
            means[s] = frame.iloc[:, pick].mean(axis=1)
        return means[late] - sum(means[p] for p in parts_early) / len(parts_early), means

    observed, means = gradient(None)
    expressed = pd.concat([means[late], *[means[p] for p in parts_early]], axis=1).max(axis=1).ge(REFERENCE_LEVEL)
    error_variance = float(sum((by_segment[s].var(axis=1, ddof=1) / by_segment[s].shape[1] * (1 if s == late else 1 / len(parts_early) ** 2))
                               for s in {late, *parts_early})[expressed].mean())
    return {'gradient': observed[expressed], 'draw': lambda rng: gradient(rng)[0][expressed],
            'error_variance': error_variance, 'n_donors': int(min(f.shape[1] for f in by_segment.values()))}


def reference_from_micro_sex(sex, contrast):
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    table = micro_sex[sex]
    early_value = table[['S1', 'S2']].mean(axis=1) if early == 'early' else table[early]
    gradient = table[late] - early_value
    expressed = pd.concat([table[late], early_value], axis=1).max(axis=1).ge(REFERENCE_LEVEL)
    return {'gradient': gradient[expressed], 'draw': lambda rng: gradient[expressed], 'error_variance': np.nan, 'n_donors': 0}



## 5 · Part 1 (D2): gradients with sampling noise for every dataset

In [ ]:
from pseudospace import zonation_reliability as reliability_module
from pseudospace.zonation_reliability import paired_gradient, unpaired_gradient, disattenuated, conservation_index, confirm_flat

D2_CONTRASTS = ('S2-S1', 'S3-early', 'S3c-early')


def ours_gradient(contrast, specimens):
    late, early = CONTRASTS[contrast]
    expressed = set(gene_space[count_filter(ours_counts, specimens, [late, early]).to_numpy()])
    return paired_gradient(ours[[(contrast, n) for n in specimens]].T.reset_index(drop=True)), expressed


def donor_gradient(count_frame, totals, donors, contrast):
    per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
    keep = level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()
    return paired_gradient(per_donor.loc[:, keep]), set(keep.index[keep.to_numpy()])


def census_human_gradient():
    donors = census_h_sex.index
    late = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'S3')], census_h_totals[(d, 'S3')]) for d in donors}).T
    early = pd.DataFrame({d: log2_cpm(census_h_counts[(d, 'early')], census_h_totals[(d, 'early')]) for d in donors}).T
    keep = np.maximum(late.mean(), early.mean()).ge(REFERENCE_LEVEL) & (late - early).notna().all()
    return paired_gradient((late - early).loc[:, keep]), set(keep.index[keep.to_numpy()])


def segment_replicates_gradient(by_segment, contrast):
    """Unpaired replicate gradient from per-segment replicate frames (rows = replicates)."""
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    if early == 'early':
        parts = [by_segment['S1'], by_segment['S2']]
        mean = by_segment[late].mean() - (parts[0].mean() + parts[1].mean()) / 2
        noise = by_segment[late].var(ddof=1) / len(by_segment[late]) + sum(p.var(ddof=1) / len(p) for p in parts) / 4
        level = pd.concat([by_segment[late].mean(), (parts[0].mean() + parts[1].mean()) / 2], axis=1).max(axis=1)
    else:
        mean, noise = unpaired_gradient(by_segment[late], by_segment[early])
        level = pd.concat([by_segment[late].mean(), by_segment[early].mean()], axis=1).max(axis=1)
    keep = level.ge(REFERENCE_LEVEL) & mean.notna()
    return (mean, noise), set(keep.index[keep.to_numpy()])


micro_male_segments = {s: micro_male[[c for c in micro_male.columns if c[0] == s]].T.reset_index(drop=True) for s in ('S1', 'S2', 'S3')}
rat_table = pd.read_csv(external / 'segment_microdissection' / 'GSE56743_RPKM.without.PAseq.txt.gz', sep='\t')
rat_table = rat_table.drop_duplicates('Symbol')
rat_table.index = rat_table.Symbol.astype(str).str.upper()
rat_table = rat_table[~rat_table.index.duplicated()]
upper_to_gene = pd.Series(gene_space, index=gene_space.str.upper())
upper_to_gene = upper_to_gene[~upper_to_gene.index.duplicated()]
rat_mapped = rat_table.index.intersection(upper_to_gene.index)
rat_mapped_fraction = len(rat_mapped) / len(gene_space)
rat_segments = {}
for seg_name in ('S1', 'S2', 'S3'):
    columns = [c for c in rat_table.columns if re.fullmatch(rf'{seg_name}_\d+', str(c))]
    values = np.log2(rat_table.loc[rat_mapped, columns].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1)
    values.index = upper_to_gene.loc[rat_mapped].to_numpy()
    rat_segments[seg_name] = values.T.reset_index(drop=True).reindex(columns=gene_space)
print(f'Rat symbols matched to our gene space: {len(rat_mapped):,} ({rat_mapped_fraction:.0%})')


def micro_sex_gradient(sex, contrast):
    ref = reference_from_micro_sex(sex, contrast)
    mean = ref['gradient']
    return (mean, pd.Series(np.nan, index=mean.index)), set(mean.index)


def datasets_for(contrast):
    human_contrast = 'S3-early' if contrast == 'S3c-early' else contrast
    out = {('human', 'ours'): ours_gradient(contrast, expected['human']),
           ('mouse', 'ours'): ours_gradient(contrast, expected['mouse']),
           ('human', 'Lake healthy'): donor_gradient(lake_counts, lake_totals, healthy_lake, human_contrast),
           ('mouse', 'snRNA male'): donor_gradient(sn_counts, sn_totals, male_sn, human_contrast),
           ('mouse', 'snRNA female'): donor_gradient(sn_counts, sn_totals, female_sn, human_contrast),
           ('mouse', 'micro male GSE150338'): segment_replicates_gradient(micro_male_segments, human_contrast),
           ('mouse', 'micro male GSE212213'): micro_sex_gradient('male', human_contrast),
           ('mouse', 'micro female GSE212213'): micro_sex_gradient('female', human_contrast),
           ('rat', 'micro GSE56743'): segment_replicates_gradient(rat_segments, human_contrast)}
    if contrast != 'S2-S1':
        out[('human', 'Census human')] = census_human_gradient()
    return out


healthy_lake = [d for d in lake_donors if lake_disease[d] == 'normal']
male_sn, female_sn = sn_sex.index[sn_sex.eq('male')], sn_sex.index[sn_sex.eq('female')]
datasets = {contrast: datasets_for(contrast) for contrast in D2_CONTRASTS}

### All pairwise noise-corrected correlations

"S3c-early" differs from "S3-early" only in our mouse, whose S3 is restricted to cortical-like
structures. Every other dataset uses its own S3.

In [ ]:
pair_rows = []
for contrast, data in datasets.items():
    keys = list(data)
    for i, a in enumerate(keys):
        for b in keys[i + 1:]:
            genes = pd.Index(sorted(data[a][1] & data[b][1]))
            if len(genes) < 200:
                continue
            species_pair = '–'.join(sorted({a[0], b[0]})) if a[0] != b[0] else f'within {a[0]}'
            result = disattenuated(data[a][0], data[b][0], genes)
            pair_rows.append({'contrast': contrast, 'comparison': species_pair, 'dataset_a': f'{a[0]}: {a[1]}',
                              'dataset_b': f'{b[0]}: {b[1]}', **result})
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(tables / 'pairwise_gradient_correlations.csv', index=False)
display(pairs[pairs.contrast.ne('S3c-early') | pairs.dataset_a.str.contains('mouse: ours') | pairs.dataset_b.str.contains('mouse: ours')]
        .round(3).sort_values(['contrast', 'comparison', 'r_corrected']))

### Conservation index

- **Primary:** our human and mouse, with ceilings r*(our human, Lake) and r*(our mouse, mouse
  snRNA male).
- **External-only:** Lake against mouse snRNA male, with the same ceilings.
- **Alternative ceilings:** Lake against Census human (S3 contrasts only), and mouse snRNA male
  against male microdissection.

In [ ]:
index_rows = []
for contrast, data in datasets.items():
    h, m, l, s = data[('human', 'ours')], data[('mouse', 'ours')], data[('human', 'Lake healthy')], data[('mouse', 'snRNA male')]
    genes = pd.Index(sorted(h[1] & m[1] & l[1] & s[1]))
    index_rows.append({'contrast': contrast, 'version': 'primary: ours human vs ours mouse',
                       **conservation_index(h[0], m[0], l[0], s[0], genes, n_boot=N_BOOT, seed=SEED)})
    index_rows.append({'contrast': contrast, 'version': 'external: Lake vs mouse snRNA male',
                       **conservation_index(l[0], s[0], h[0], m[0], genes, n_boot=N_BOOT, seed=SEED)})
    micro = data[('mouse', 'micro male GSE150338')]
    if ('human', 'Census human') in data:
        c = data[('human', 'Census human')]
        genes_alt = pd.Index(sorted(l[1] & s[1] & c[1] & micro[1]))
        index_rows.append({'contrast': contrast, 'version': 'external, alternative ceilings: Lake vs snRNA male (Census human, microdissection)',
                           **conservation_index(l[0], s[0], c[0], micro[0], genes_alt, n_boot=N_BOOT, seed=SEED)})
conservation = pd.DataFrame(index_rows)
conservation.to_csv(tables / 'conservation_index.csv', index=False)
display(conservation.round(3))

primary_index = conservation[conservation.version.str.startswith('primary') & conservation.contrast.isin(['S2-S1', 'S3-early'])]
if (primary_index.ci_high < INDEX_STRONG).all():
    D2_WORDING = 'largely different genes (stands)'
elif (primary_index.ci_low >= INDEX_PARTIAL).all():
    D2_WORDING = 'largely conserved; apparent divergence is a cross-dataset ceiling'
else:
    D2_WORDING = 'partly conserved'
rat_reliability = pairs[pairs.dataset_b.eq('rat: micro GSE56743') | pairs.dataset_a.eq('rat: micro GSE56743')].groupby('contrast').apply(
    lambda d: float(np.where(d.dataset_b.eq('rat: micro GSE56743'), d.reliability_b, d.reliability_a).mean()))
RAT_USABLE = bool((rat_reliability >= RAT_RELIABILITY).all() and rat_mapped_fraction >= RAT_MAPPED)
print(f'D2 wording: {D2_WORDING}')
print(f'Rat: reliability {rat_reliability.round(3).to_dict()}, mapped {rat_mapped_fraction:.0%} → '
      + ('usable' if RAT_USABLE else 'excluded as unreliable; withdraw rat-based arguments'))

## 6 · Part 2 (D3): symmetric classification

These are notebook 40's moderated contrasts and calls, with the additional settings of the
protocol.

In [ ]:
from pseudospace.zonation_classes import moderated_contrast, zonation_calls, confirm, cross_species_class, union_class

CLASS_CONTRASTS = ('S2-S1', 'S3c-early', 'S3-early')


def species_level(pb, pb_library, specimens, parts):
    return pd.concat([log2_cpm(pb[(n, p)], pb_library[(n, p)]) for n in specimens for p in parts], axis=1).mean(axis=1)


def reference_table(count_frame, totals, donors, contrast):
    per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
    per_donor = per_donor.loc[:, level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()]
    return moderated_contrast(per_donor, level.mean())


tables_ours, tables_ref = {}, {}
for contrast in CLASS_CONTRASTS:
    late, early = CONTRASTS[contrast]
    both = gene_space[(count_filter(ours_counts, expected['mouse'], [late, early])
                       & count_filter(ours_counts, expected['human'], [late, early])).to_numpy()]
    for species in ('mouse', 'human'):
        replicates = ours.loc[both, [(contrast, n) for n in expected[species]]].T.reset_index(drop=True)
        tables_ours[(contrast, species)] = moderated_contrast(replicates, species_level(ours_counts, ours_library, expected[species], [late, early]))
    tables_ref[(contrast, 'mouse male')] = reference_table(sn_counts, sn_totals, male_sn, contrast)
    tables_ref[(contrast, 'mouse female')] = reference_table(sn_counts, sn_totals, female_sn, contrast)
    tables_ref[(contrast, 'human')] = reference_table(lake_counts, lake_totals, healthy_lake, contrast)
micro_gradient = {(sex, c): (reference_from_micro_male(c) if sex == 'male' else reference_from_micro_sex('female', c))['gradient']
                  for sex in ('male', 'female') for c in CLASS_CONTRASTS}


def noise_corrected_amplitude(contrast):
    genes = tables_ours[(contrast, 'mouse')].index

    def signal(specimens):
        reps = ours.loc[genes, [(contrast, n) for n in specimens]].to_numpy()
        return reps.mean(axis=1).var(ddof=1) - reps.var(axis=1, ddof=1).mean() / reps.shape[1]
    return float(np.sqrt(max(signal(expected['human']), 0) / signal(expected['mouse'])))


amplitude = {c: noise_corrected_amplitude(c) for c in CLASS_CONTRASTS}
print('Noise-corrected human ÷ mouse amplitude:', {k: round(v, 3) for k, v in amplitude.items()})

SETTINGS = {
    'v1 (notebook 40 primary)': dict(scale=False, flat_external=False, confirm='snRNA male'),
    'v1 + A6 female snRNA': dict(scale=False, flat_external=False, confirm='snRNA female'),
    'v1 + A7 female microdissection': dict(scale=False, flat_external=False, confirm='micro female'),
    'A3b amplitude-matched': dict(scale=True, flat_external=False, confirm='snRNA male'),
    'A8 external flat confirmation': dict(scale=False, flat_external=True, confirm='snRNA male'),
    'symmetric (A3b + A8)': dict(scale=True, flat_external=True, confirm='snRNA male'),
    'symmetric + A6 female snRNA': dict(scale=True, flat_external=True, confirm='snRNA female'),
    'symmetric + A7 female microdissection': dict(scale=True, flat_external=True, confirm='micro female')}


def mouse_calls_for(contrast, effect, margin, confirm_by):
    calls, _ = zonation_calls(tables_ours[(contrast, 'mouse')], alpha=ALPHA, effect=effect, margin=margin)
    if confirm_by.startswith('snRNA'):
        return confirm(calls, tables_ref[(contrast, 'mouse ' + confirm_by.split()[1])], alpha=CONFIRM_ALPHA)
    ref = micro_gradient[(confirm_by.split()[1], contrast)].reindex(calls.index)
    ok = ((calls == 'up') & (ref >= EFFECT)) | ((calls == 'down') & (ref <= -EFFECT))
    return calls.where(~calls.isin(['up', 'down']) | ok.fillna(False), 'indeterminate')


def classify(setting):
    spec = SETTINGS[setting]
    frame = {}
    for contrast in CLASS_CONTRASTS:
        scale = amplitude[contrast] if spec['scale'] else 1.
        mouse_calls = mouse_calls_for(contrast, EFFECT, MARGIN, spec['confirm'])
        human_calls, _ = zonation_calls(tables_ours[(contrast, 'human')], alpha=ALPHA, effect=EFFECT * scale, margin=MARGIN * scale)
        human_calls = confirm(human_calls, tables_ref[(contrast, 'human')], alpha=CONFIRM_ALPHA)
        classes = pd.Series([cross_species_class(a, b) for a, b in zip(mouse_calls, human_calls)], index=mouse_calls.index)
        if spec['flat_external']:
            mouse_ref = tables_ref[(contrast, 'mouse female' if spec['confirm'] == 'snRNA female' else 'mouse male')]
            classes = confirm_flat(classes, mouse_calls, human_calls, mouse_ref, tables_ref[(contrast, 'human')],
                                   alpha=CONFIRM_ALPHA, effect=EFFECT)
        frame[contrast], frame[contrast + ' mouse call'], frame[contrast + ' human call'] = classes, mouse_calls, human_calls
    frame = pd.DataFrame(frame)
    frame['union S2-S1 + S3c-early'] = union_class(frame[['S2-S1', 'S3c-early']])
    return frame


probe = pd.read_csv(probe_file).drop_duplicates('mouse_symbol').set_index('mouse_symbol')
balanced = set(probe.index[probe.probe_balance.eq('balanced')])
classified = {setting: classify(setting) for setting in SETTINGS}
ORDER = ['conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither']
count_rows = []
for setting, frame in classified.items():
    frame_b = frame[frame.index.isin(balanced)]
    for contrast in (*CLASS_CONTRASTS, 'union S2-S1 + S3c-early'):
        counts_ = frame_b[contrast].value_counts().reindex(ORDER).fillna(0).astype(int)
        count_rows.append({'setting': setting, 'contrast': contrast, **counts_.to_dict(),
                           'mouse-only ÷ human-only': counts_['mouse-only'] / max(counts_['human-only'], 1)})
class_counts = pd.DataFrame(count_rows)
class_counts.to_csv(tables / 'class_counts_by_setting_and_contrast.csv', index=False)
display(class_counts.set_index(['setting', 'contrast'])[['conserved', 'reversal', 'mouse-only', 'human-only', 'mouse-only ÷ human-only']].round(2))
symmetric = classified['symmetric (A3b + A8)']
symmetric.assign(probe_balanced=symmetric.index.isin(balanced)).to_csv(tables / 'gene_classes_symmetric.csv')

ratio_sym = class_counts[class_counts.setting.eq('symmetric (A3b + A8)')].set_index('contrast')['mouse-only ÷ human-only']
if ratio_sym['S2-S1'] >= 2 and ratio_sym['S3c-early'] >= 2:
    D3_ASYMMETRY = 'asymmetry supported'
elif ratio_sym['S2-S1'] >= 2:
    D3_ASYMMETRY = 'asymmetry confined to S2 − S1'
else:
    D3_ASYMMETRY = 'no consistent asymmetry'
print('D3 decision:', D3_ASYMMETRY, ratio_sym.round(2).to_dict())

strong_rows = {}
for setting in ('v1 (notebook 40 primary)', 'symmetric (A3b + A8)'):
    frame = classified[setting]
    genes_ = frame.index[frame['S2-S1'].eq('mouse-only') & frame.index.isin(balanced)]
    m, h = tables_ours[('S2-S1', 'mouse')]['mean'].loc[genes_], tables_ours[('S2-S1', 'human')]['mean'].loc[genes_]
    strong_rows[setting] = {'S2 − S1 mouse-only genes': len(genes_), '|mouse| ≥ 1.5 and |human| ≤ 0.3': int(((m.abs() >= 1.5) & (h.abs() <= .3)).sum()),
                            '|mouse| ≥ 1 ': int((m.abs() >= 1).sum()), 'median |mouse|': float(m.abs().median())}
strong_mouse_only = pd.DataFrame(strong_rows).T
strong_mouse_only.to_csv(tables / 'mouse_only_gradient_distribution.csv')
display(strong_mouse_only)

## 7 · Part 3 (M7): what the robust pathways summarise

In [ ]:
import ast

final37 = pd.read_csv(upstream37 / 'tables' / 'table_S2_pathways.csv', index_col=0)
robust_ids = final37.index[final37.robust.astype(bool)]
gene_stats12 = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv', index_col=0)
contributing = set(gene_stats12.index[gene_stats12.T_spatial.ge(gene_stats12.T_spatial.quantile(.9))])
coverage = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'pathway_coverage.csv').set_index('pathway_id')
flattening39 = pd.read_csv(results_root / 'pt_zonation_amplitude' / 'tables' / 'robust_pathway_flattening_classes.csv', index_col=0)


def member_category(gene):
    row = symmetric.loc[gene] if gene in symmetric.index else None
    if row is None:
        return 'not classifiable'
    categories = []
    for contrast in ('S2-S1', 'S3c-early'):
        klass = row.get(contrast)
        if not isinstance(klass, str):
            continue
        mouse_zonated = row[contrast + ' mouse call'] in ('up', 'down')
        human_zonated = row[contrast + ' human call'] in ('up', 'down')
        if klass in ('conserved', 'reversal'):
            return 'shared'
        if mouse_zonated and not human_zonated:
            categories.append('mouse-led')
        elif human_zonated and not mouse_zonated:
            categories.append('human-led')
    if 'mouse-led' in categories and 'human-led' in categories:
        return 'mixed'
    return categories[0] if categories else 'not zonated'


pathway_rows = []
for p in robust_ids:
    members = [g for g in ast.literal_eval(coverage.loc[p, 'genes_present']) if g in contributing]
    categories = pd.Series([member_category(g) for g in members], dtype=object).value_counts()
    zonated = int(categories.reindex(['mouse-led', 'human-led', 'shared', 'mixed']).fillna(0).sum())
    shares = (categories / zonated) if zonated else categories * 0
    if zonated < 3:
        label = 'unresolved'
    elif shares.get('mouse-led', 0) >= .5:
        label = 'summary of mouse zonation'
    elif shares.get('shared', 0) >= .5:
        label = 'shared'
    elif shares.get('human-led', 0) >= .5:
        label = 'human-led'
    else:
        label = 'mixed'
    probe_ok = bool(final37.loc[p, 'probe_equal_pass']) if 'probe_equal_pass' in final37 else np.nan
    flat39 = flattening39['class'].get(p, 'not tested')
    pathway_rows.append({'pathway_id': p, 'name': final37.loc[p, 'name'], 'core': bool(final37.loc[p, 'core']),
                         'contributing_members': len(members), 'zonated_contributing': zonated,
                         **{f'n_{k}': int(categories.get(k, 0)) for k in ('mouse-led', 'human-led', 'shared', 'mixed', 'not zonated', 'not classifiable')},
                         'category': label, 'notebook39_class': flat39, 'probe_equal_pass': probe_ok,
                         'q_probe_equal': final37.loc[p, 'q_probe_equal'] if 'q_probe_equal' in final37 else np.nan,
                         'keep_beyond_flattening': (flat39 == 'beyond flattening') and probe_ok})
pathway_reframe = pd.DataFrame(pathway_rows).set_index('pathway_id')
pathway_reframe.to_csv(tables / 'robust_pathways_reframed.csv')
display(pathway_reframe[['name', 'contributing_members', 'zonated_contributing', 'n_mouse-led', 'n_human-led', 'n_shared',
                         'category', 'notebook39_class', 'probe_equal_pass']].sort_values('category'))
display(pathway_reframe.category.value_counts())
beyond = pathway_reframe[pathway_reframe.notebook39_class.eq('beyond flattening')]
display(beyond[['name', 'probe_equal_pass', 'q_probe_equal', 'keep_beyond_flattening']])
probe_sensitive = pathway_reframe.index[~pathway_reframe.probe_equal_pass.astype(bool)]
print(f'Probe-sensitive robust pathways: {len(probe_sensitive)}: ' + '; '.join(pathway_reframe.loc[probe_sensitive, 'name']))

### Reading the results

- **D2.** Same-species, cross-dataset ceilings are high for mouse and moderate for human. The
  cross-species correlations are far below both, so the conservation index stays well under
  0.5 for every contrast and version. Gene-level zonation is therefore largely species-specific,
  both in our data and in external atlases alone, even after allowing for cross-dataset ceilings.
- **Rat.** The rat microdissection data fail the reliability rule, so rat-based arguments are
  withdrawn.
- **D3.** Once human thresholds are matched to the lower human amplitude, and flat calls must be
  flat in the external atlas, the excess of mouse-only over human-only genes disappears or
  reverses. The notebook 40 asymmetry was a threshold effect of lower human amplitude. The
  conserved class becomes the largest zonated class. The species-specificity shows up as a
  genome-wide low correlation, not as a large set of confidently species-only genes.
- **M7.** Most robust pathways summarise mouse zonation that human PT does not share. None is
  human-led. Two of the four "beyond flattening" pathways fail the equal-probe check and are
  dropped.

## 8 · Figures

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 8.5,
                     'axes.titlesize': 9.5, 'axes.titleweight': 'bold', 'pdf.fonttype': 42})
HUMAN, MOUSE, INK, NULL = '#D55E00', '#0072B2', '#333333', '#9A9A9A'
COLORS = {'within human': HUMAN, 'within mouse': MOUSE, 'human–mouse': '#009E73', 'mouse–rat': '#CC79A7', 'human–rat': '#999999'}


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=220)


fig, axes = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={'width_ratios': [1.3, 1, 1.2]})
ax = axes[0]
plot_pairs = pairs[pairs.contrast.isin(['S2-S1', 'S3-early'])].dropna(subset=['r_corrected'])
for k, contrast in enumerate(['S2-S1', 'S3-early']):
    block = plot_pairs[plot_pairs.contrast.eq(contrast)]
    for j, comparison in enumerate(['within human', 'within mouse', 'human–mouse', 'mouse–rat']):
        values = block.loc[block.comparison.eq(comparison), 'r_corrected']
        x = k * 5 + j + np.random.default_rng(j).uniform(-.15, .15, len(values))
        ax.scatter(x, values.clip(-.2, 1.2), color=COLORS[comparison], s=22, label=comparison if k == 0 else None)
ax.set_xticks([1.5, 6.5], ['S2 − S1', 'S3 − early'])
ax.axhline(0, color=NULL, lw=.6)
ax.set_ylabel('Noise-corrected correlation of gene gradients')
ax.legend(frameon=False, fontsize=7)
ax.set_title('a · Same-species ceilings and cross-species agreement', loc='left')
ax = axes[1]
plot_index = conservation[conservation.contrast.isin(['S2-S1', 'S3-early', 'S3c-early'])]
yy = np.arange(len(plot_index))[::-1]
ax.errorbar(plot_index['index'], yy, xerr=[plot_index['index'] - plot_index.ci_low, plot_index.ci_high - plot_index['index']], fmt='o', color=INK)
ax.set_yticks(yy, [f'{c} · {v.split(":")[0]}' for c, v in zip(plot_index.contrast, plot_index.version)], fontsize=7)
for value in (INDEX_STRONG, INDEX_PARTIAL, 1):
    ax.axvline(value, color=NULL, ls='--' if value < 1 else '-', lw=.8)
ax.set_xlabel('Conservation index (cross ÷ within-species ceiling)')
ax.set_title('b · Conservation index', loc='left')
ax = axes[2]
show = class_counts[class_counts.setting.isin(['v1 (notebook 40 primary)', 'symmetric (A3b + A8)', 'symmetric + A6 female snRNA',
                                               'symmetric + A7 female microdissection']) & class_counts.contrast.isin(CLASS_CONTRASTS)]
labels_ = [f'{s.split(" (")[0]} · {c}' for s, c in zip(show.setting, show.contrast)]
yy = np.arange(len(show))[::-1]
ax.barh(yy + .2, show['mouse-only'], height=.38, color=MOUSE, label='mouse-only')
ax.barh(yy - .2, show['human-only'], height=.38, color=HUMAN, label='human-only')
ax.set_yticks(yy, labels_, fontsize=6.5)
ax.set_xlabel('Probe-balanced genes')
ax.legend(frameon=False, fontsize=7)
ax.set_title('c · Species-only classes by setting and contrast', loc='left')
fig.tight_layout()
save(fig, 'fig_revision_ceiling_and_classes')
plt.show()
plt.close(fig)

## 9 · Decisions and run record

In [ ]:
decisions = pd.Series({'D2 wording': D2_WORDING, 'rat usable': RAT_USABLE, 'D3 asymmetry': D3_ASYMMETRY,
                       'robust pathways: summary of mouse zonation': int(pathway_reframe.category.eq('summary of mouse zonation').sum()),
                       'robust pathways: shared': int(pathway_reframe.category.eq('shared').sum()),
                       'robust pathways: human-led': int(pathway_reframe.category.eq('human-led').sum()),
                       'robust pathways: mixed or unresolved': int(pathway_reframe.category.isin(['mixed', 'unresolved']).sum()),
                       'beyond-flattening pathways kept after probe check': ', '.join(pathway_reframe.loc[pathway_reframe.keep_beyond_flattening.astype(bool), 'name']),
                       'beyond-flattening pathways dropped (probe-sensitive)': ', '.join(beyond.loc[~beyond.keep_beyond_flattening.astype(bool), 'name'])})
decisions.to_csv(tables / 'decisions.csv')
display(decisions.to_frame('value'))
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'labels': str(label_file.name) if label_file else 'reviewed (notebook 13)',
            'decisions': decisions.astype(str).to_dict(), 'amplitude': amplitude,
            'conservation_index': conservation.round(4).to_dict(orient='records'),
            'rat_reliability': rat_reliability.to_dict(), 'rat_mapped_fraction': rat_mapped_fraction}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps(manifest['decisions'], indent=2))